In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torchvision.datasets import MNIST

In [2]:
from torchvision.transforms import transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.6) , (0.6))
])
trainset = MNIST(root = "./data" , train = True , transform = transform , download = True)
testset = MNIST(root = "./data" , train = False , transform = transform , download = True)

In [3]:
from torch.utils.data import DataLoader

trainLoader = DataLoader(trainset , batch_size = 64 , shuffle = True)
testLoader = DataLoader(testset , batch_size = 64 , shuffle = False)

In [4]:
img , label = trainset[0]
print(img.shape)
print(label)

torch.Size([1, 28, 28])
5


In [5]:
import torch.optim as optim

## Build the CNN

In [6]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN , self).__init__()

        self.conv_layer = nn.Sequential(
            nn.Conv2d(1 , 32 , kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2 , 2),

            nn.Conv2d(32 , 64 , kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2 , 2),

            nn.Conv2d(64 , 128 , kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2 , 2),

        )

        self.fc_layer = nn.Sequential(
            nn.Linear(3 * 3 *128 , 256),
            nn.ReLU(),
            nn.Linear(256 , 10)
        )


    def forward(self , x):
        x = self.conv_layer(x)
        x = x.view(x.size(0) , -1)
        x = self.fc_layer(x)

        return x      
        

In [7]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [8]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images , labels in trainLoader:
        optimizer.zero_grad()

        outputs = model.forward(images)
        loss = criterion(outputs , labels)
        loss.backward()
        optimizer.step()  #parameter update

        running_loss += loss.item()

    print(f"{epoch + 1} / {epochs} and loss = {running_loss/len(trainLoader)}")

1 / 10 and loss = 0.15506577914665595
2 / 10 and loss = 0.040633954492363226
3 / 10 and loss = 0.02936291758709876
4 / 10 and loss = 0.023413428025518004
5 / 10 and loss = 0.01797077953817477
6 / 10 and loss = 0.01500575253515275
7 / 10 and loss = 0.013413696511716946
8 / 10 and loss = 0.010441921385761832
9 / 10 and loss = 0.011134732485728462
10 / 10 and loss = 0.006766178893021187


In [9]:
correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images , labels in testLoader:
        outputs = model.forward(images)
        _  , predicted = torch.max(outputs , 1)
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)


    print(f"accuracy : {correct_labels /total_labels *100 }")

    

accuracy : 98.7


## Confusion matrix for CNN

In [36]:
from sklearn.metrics import confusion_matrix

y_true = []
y_pred = []

model.eval()
with torch.no_grad():

    for images , labels in testLoader:
        outputs = model(images)
        _ , predicted = torch.max(outputs , 1)

        
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

# print(type(y_true[0]), type(y_pred[0]))
# print(len(y_true), len(y_pred))

print("Confusion matrix : " , confusion_matrix(y_true , y_pred))

Confusion matrix :  [[ 977    1    0    0    0    0    0    1    1    0]
 [   0 1135    0    0    0    0    0    0    0    0]
 [   1    2 1018    0    0    0    0    8    3    0]
 [   1    1    2  993    1   10    0    1    1    0]
 [   0    1    1    0  975    0    4    0    0    1]
 [   1    2    0    2    0  882    3    0    1    1]
 [   2    3    0    0    0    1  951    0    1    0]
 [   0   18    2    0    3    0    0  999    3    3]
 [   2    2    1    0    0    0    2    0  966    1]
 [   1    5    0    0   15    6    0    2    6  974]]


## RNN Implementation 

In [40]:
class RNN(nn.Module):
    def __init__(self , in_size = 28 , hidden_size = 64 , num_layers = 2):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #Rnn layer
        self.rnn = nn.RNN(in_size , hidden_size , num_layers, batch_first = True)

        #fc layer
        self.fc_layer = nn.Linear(hidden_size , 10)

    def forward(self , x):
        out , _ = self.rnn(x)
        out = self.fc_layer(out[: , -1 , :])

        return out

In [41]:
model_rnn = RNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_rnn.parameters())

In [ ]:
epochs = 10

for epoch in range(epochs):
    model_rnn.train()

    for images , labels in trainLoader:
    
        optimizer.zero_grad()

        outputs = model_rnn(images.squeeze())
        loss = criterion(outputs ,labels)
        loss.backward()
        optimizer.step()

    print(f"{epoch + 1}/{epochs} and loss = {loss.item()}")

1/10 and loss = 0.21374738216400146
2/10 and loss = 0.24704767763614655


In [33]:
correct_values = 0.0
total_values = 0.0


model.eval()
with torch.no_grad():
    for images , labels in testLoader:

        outputs = model_rnn(images.squeeze())
        _  , predicted = torch.max(outputs , 1)

        correct_values += (predicted == labels).sum().item()
        total_values += labels.size(0)

    print(f"accuracy : {correct_values /total_values *100 }")

accuracy : 96.89999999999999


In [35]:
from sklearn.metrics import confusion_matrix

y_true = []
y_pred = []

model.eval()
with torch.no_grad():

    for images , labels in testLoader:
        outputs = model_rnn(images.squeeze())
        predicted = torch.argmax(outputs , 1)

        
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

# print(type(y_true[0]), type(y_pred[0]))
# print(len(y_true), len(y_pred))

print("Confusion matrix : " , confusion_matrix(y_true , y_pred))

Confusion matrix :  [[ 966    1    6    1    1    1    2    1    1    0]
 [   0 1128    4    1    0    0    0    1    1    0]
 [   1    3 1004    3    1    2    1   10    7    0]
 [   0    0   17  961    0   18    0   12    2    0]
 [   0    0    2    0  953    3    2    1    0   21]
 [   4   11    0   11    3  848    4    1    6    4]
 [   9    3    5    0   12    6  917    0    6    0]
 [   0    3   10    0    0    0    0 1013    2    0]
 [   4    5    3    4    4    4    2    6  941    1]
 [   1    2    2    1   13    6    0   18    7  959]]
